Knowledge Distillation (Sequence) Step 1: Teacher Output

In [1]:
import ast
import json
import os
import textwrap
import time
import pandas as pd
import torch
from huggingface_hub import list_repo_files, hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM

teacher_name = "codellama/CodeLlama-7b-Instruct-hf"
dataset_repo = "code-search-net/code_search_net"
out_path = "/content/teacher_outputs.jsonl"

n_prompts = 400          # functions sent to the teacher
max_code_tokens = 220    # skip long functions (keeps prompts short and memory safe)
max_new_tokens = 384     # room for the full function plus a Google-style docstring
batch_size = 4           # halves automatically on a GPU out-of-memory error
seed = 0

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. Set the Colab runtime to T4.")

tokenizer = AutoTokenizer.from_pretrained(teacher_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"          # required for batched generation

# --- 1. Data: functions whose existing docstring is NOT Google style (no "Args:"),
#        with at least one parameter and a return value, so Args/Returns are meaningful.
files = [f for f in list_repo_files(dataset_repo, repo_type="dataset", revision="refs/convert/parquet")
         if f.startswith("python/") and f.endswith(".parquet") and ("valid" in f or "test" in f)]
paths = [hf_hub_download(dataset_repo, f, repo_type="dataset", revision="refs/convert/parquet")
         for f in files]
df = pd.concat(pd.read_parquet(p, columns=["whole_func_string", "func_documentation_string"])
               for p in paths)
df = df[~df["func_documentation_string"].str.contains("Args:", regex=False)
        & ~df["whole_func_string"].str.contains("below_zero", regex=False)]
print(f"Non-Google candidates: {len(df):,} functions from {len(files)} parquet files")

def usable(source):
    """Return (dedented source, function name) if the function suits the task, else None."""
    src = textwrap.dedent(source)
    try:
        fn = ast.parse(src).body[0]
    except (SyntaxError, IndexError):
        return None
    if not isinstance(fn, (ast.FunctionDef, ast.AsyncFunctionDef)) or len(fn.body) < 2:
        return None
    doc = fn.body[0]
    if not (isinstance(doc, ast.Expr) and isinstance(doc.value, ast.Constant)
            and isinstance(doc.value.value, str)):
        return None
    params = [a.arg for a in fn.args.posonlyargs + fn.args.args + fn.args.kwonlyargs
              if a.arg not in ("self", "cls")]
    returns = any(isinstance(n, ast.Return) and n.value is not None for n in ast.walk(fn))
    if not params or not returns:
        return None
    return src, fn.name

prompts = []
for source in df["whole_func_string"].sample(frac=1, random_state=seed):
    item = usable(source)
    if item is None:
        continue
    src, name = item
    if len(tokenizer(src, add_special_tokens=False)["input_ids"]) > max_code_tokens:
        continue
    prompts.append({"id": len(prompts), "name": name, "code": src})
    if len(prompts) == n_prompts:
        break
print(f"Prompts prepared: {len(prompts)}")

# --- 2. Teacher: the unchanged baseline model in FP16
model = AutoModelForCausalLM.from_pretrained(teacher_name, dtype=torch.float16, device_map="cuda")
model.eval()

prompt_template = ("Generate Google-style docstring for the following Python function: {code}. "
                   "Only output the docstring and function code.")   # P-2 wording

def generate(batch):
    texts = [tokenizer.apply_chat_template(
                 [{"role": "user", "content": prompt_template.format(code=p["code"])}],
                 add_generation_prompt=True, tokenize=False) for p in batch]
    enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    results = []
    for p, row in zip(batch, out[:, enc["input_ids"].shape[1]:]):
        finished = bool((row == tokenizer.eos_token_id).any())
        text = tokenizer.decode(row, skip_special_tokens=True)
        keep = (finished and "Args:" in text and "Returns:" in text
                and f"def {p['name']}(" in text)
        results.append({**p, "output": text, "finished": finished, "kept": keep})
    return results

def run(batch):
    try:
        return generate(batch)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(batch) == 1:
            raise
        mid = len(batch) // 2
        return run(batch[:mid]) + run(batch[mid:])

# --- 3. Generate, appending each batch to the JSONL (resumes if the file already exists)
done = set()
if os.path.exists(out_path):
    with open(out_path) as f:
        done = {json.loads(line)["id"] for line in f}
todo = sorted((p for p in prompts if p["id"] not in done), key=lambda p: len(p["code"]))
print(f"Already done: {len(done)} | to generate: {len(todo)}")

start = time.perf_counter()
batches = [todo[i:i + batch_size] for i in range(0, len(todo), batch_size)]
for b, batch in enumerate(batches, 1):
    with open(out_path, "a") as f:
        for r in run(batch):
            f.write(json.dumps(r) + "\n")
    if b % 10 == 0 or b == len(batches):
        print(f"batch {b}/{len(batches)} | {(time.perf_counter() - start) / 60:.1f} min")

with open(out_path) as f:
    rows = [json.loads(line) for line in f]
kept = sum(r["kept"] for r in rows)
print(f"\nTeacher generation time this run: {(time.perf_counter() - start) / 60:.1f} min")
print(f"Outputs: {len(rows)} | kept (complete, Google-style): {kept} ({kept / len(rows):.0%})")
print(f"Saved to {out_path}")


config.json:   0%|          | 0.00/646 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.59k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Already done: 0 | to generate: 400
batch 10/100 | 2.2 min
batch 20/100 | 4.6 min
batch 30/100 | 7.7 min
batch 40/100 | 11.1 min
batch 50/100 | 14.5 min
batch 60/100 | 17.9 min
batch 70/100 | 22.0 min
batch 80/100 | 26.1 min
batch 90/100 | 29.9 min
batch 100/100 | 34.1 min

Teacher generation time this run: 34.1 min
Outputs: 400 | kept (complete, Google-style): 140 (35%)
Saved to /content/teacher_outputs.jsonl  <- download this file as a backup
